In [3]:
import torch

torch.__version__

'2.14.0+cu130'

In [4]:
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [8]:
train_dataset = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_dataset = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()    
)

# FashionMNIST 객체
FashionMNIST 객체는 다음과 같은 형태로 이루어져 있습니다.
- 이미지 데이터
- label 데이터
- transform = ToTensor()

In [13]:
x, y = train_dataset[0]
print(type(x))
print(x.shape)

<class 'torch.Tensor'>
torch.Size([1, 28, 28])


In [18]:
train_loader = DataLoader(
    dataset=train_dataset,
    shuffle=True,
    batch_size=64,
)

test_loader = DataLoader(
    dataset=test_dataset,
    shuffle=False,
    batch_size=64
)

In [21]:
train_iter = iter(train_loader)

X, y = next(train_iter)

In [22]:
print(X.shape)
print(y.shape)
print(X.dtype)
print(y.dtype)

torch.Size([64, 1, 28, 28])
torch.Size([64])
torch.float32
torch.int64


데이터의 크기는 b=64, c=1, 28*28이며 이를 하나의 선형 모델로 변경하기 위해선 shape를 `(b=64, 1*28*28)`로 변경해야합니다.

torch의 경웨서 PIL을 ToTensor로 바꾼 형태이기 때문에 images의 타입은 float32, 라벨의 경우에는 int64 형태로 반환되는 것을 알 수 있습니다.

또한 분류 문제에서 CrossEntropyLoss의 경우에는 class label을 클래스의 인덱스로 받기 때문에 target을 보통 int64/long를 요구합니다.

In [23]:
print(X.min())
print(X.max())

tensor(0.)
tensor(1.)


In [ ]:
# start_dim=1 만 보존한 상태로 진행하기
X_flat = torch.flatten(X, start_dim=1)
print(X_flat.shape)

torch.Size([64, 784])


# PyTorch의 공식 튜토리얼 모델 만들어보기
```
입력
[batch, 1, 28, 28]

↓ Flatten

[batch, 784]

↓ Linear

[batch, 512]

↓ ReLU

[batch, 512]

↓ Linear

[batch, 512]

↓ ReLU

[batch, 512]

↓ Linear

[batch, 10]

= logits
```

In [30]:
from torch import nn

class NeuralNetwork(nn.Module):

    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()

        self.linear_relu_stack = nn.Sequential(
            # 1*28*28 사이즈 받기
            nn.Linear(28*28, 512),
            nn.ReLU(),

            nn.Linear(512, 512),
            nn.ReLU(),

            # 마지막 512 -> 10
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.linear_relu_stack(x)

In [31]:
model = NeuralNetwork()

# 임의의 한 batch_size=64인 dataset를 받아주기
logits = model(X)

# 최종적으로 선형 모델들의 마지막 반환인 10 크기로 반환 (batch size는 보존)
print(logits.shape)

torch.Size([64, 10])


In [ ]:
first_linear = model.linear_relu_stack[0]

print(first_linear)
# 512, 28*28 (실제로 전치해서 사용하기 때문에 out, in 순서로 나오게 됨)
print(first_linear.weight.shape)
# 512
print(first_linear.bias.shape)

Linear(in_features=784, out_features=512, bias=True)
torch.Size([512, 784])
torch.Size([512])


In [37]:
# 아래의 weight와 bias인 Parameter(Torch)는 내부적으로 requires_grad가 되어있는 것을 확인할 수 있다.
print(type(first_linear.weight))
print(type(first_linear.bias))

print(first_linear.weight.requires_grad)
print(first_linear.bias.requires_grad)

<class 'torch.nn.parameter.Parameter'>
<class 'torch.nn.parameter.Parameter'>
True
True


In [40]:
first_relu = model.linear_relu_stack[1]
print(type(first_relu))
print(hasattr(first_relu, "weight"))
print(hasattr(first_relu, "bias"))

<class 'torch.nn.modules.activation.ReLU'>
False
False


NeuralNetwork 객체에는 내부적으로
- flatten: nn.Flatten
- linear_relu_stack: nn.Sequential
    - [0] Linear
        - weight: Parameter/Tensor + requires_grad
        - bias: Parameter/Tensor + requires_grad
    - [1] ReLU

가 존재하는 것을 확인할 수 있습니다.

In [48]:
# Linear 연산을 직접 재현
# batch를 먼저 flatten
X_flat = model.flatten(X)

print(X_flat.shape)

torch.Size([64, 784])


In [ ]:
with torch.no_grad(): # 자동 미분 DAG 추적을 끄는 컨텍스트 inference_mode 보다는 약함
    output_module = first_linear(X_flat)

In [50]:
with torch.no_grad():   # first_linear의 가중치를 그대로 활용하되 전치해서 행렬곱
    output_manual = X_flat @ first_linear.weight.T + first_linear.bias

In [55]:
print(output_module.shape)
print(output_manual.shape)


print(torch.allclose(output_module, output_manual, rtol=1e-4, atol=1e-6))
print(torch.allclose(output_module, output_manual))
print(torch.abs(output_module - output_manual).sum())

torch.Size([64, 512])
torch.Size([64, 512])
True
False
tensor(0.0002)


Torch의 allclose는 원소에 대해 abs(a - b) <= atol + rtol*abs(b) 를 검사함.

각각 relative tolerance와 absolute tolerance를 이용해서 두 값의 차이가 절대 허용에 두번째 값의 배율을 먹여서 차이를 검사하게 됩니다. 두번째 인자가 작아질수록 그 차이의 요구사항은 더욱 빡빡해집니다.